# Ingestion exploration

In [91]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [92]:
import sys
sys.path.append("..")

from pipeline import extract, normalize, chunk, embed, store, config

pdfs = sorted(config.PDF_DIR.glob("*.pdf"))
pdfs

[WindowsPath('E:/University of Kerman/Term 10/Natural Language Processing/Final Project/ukask/data/pdfs/ainname karshenasi phd msc 1397.pdf'),
 WindowsPath('E:/University of Kerman/Term 10/Natural Language Processing/Final Project/ukask/data/pdfs/shivename 1402.pdf'),
 WindowsPath('E:/University of Kerman/Term 10/Natural Language Processing/Final Project/ukask/data/pdfs/خلاصه قوانین آموزشی.pdf'),
 WindowsPath('E:/University of Kerman/Term 10/Natural Language Processing/Final Project/ukask/data/pdfs/دستورالعمل بررسی موارد غیبت پزشکی.pdf'),
 WindowsPath('E:/University of Kerman/Term 10/Natural Language Processing/Final Project/ukask/data/pdfs/راهنمای شرکت در امتحانات پایان ترم.pdf'),
 WindowsPath('E:/University of Kerman/Term 10/Natural Language Processing/Final Project/ukask/data/pdfs/مجموعه مقررات آموزشی 1402.pdf'),
 WindowsPath('E:/University of Kerman/Term 10/Natural Language Processing/Final Project/ukask/data/pdfs/مقررات-آموزشی-ابلاغی-1402.pdf')]

## Extraction

Checking which pages came back as text versus OCR. with scanned vs text detection.

### OCR

In [100]:
pdf_path = pdfs[1]
raw_pages = extract.extract_pdf(pdf_path)
for p in raw_pages:
    print(p["page"], "OCR" if p["was_ocr"] else "native", len(p["text"]), "chars")

shivename 1402.pdf: 100%|██████████| 25/25 [00:29<00:00,  1.18s/page, OCR, slower]

1 OCR 0 chars
2 OCR 2116 chars
3 OCR 2377 chars
4 OCR 1669 chars
5 OCR 1344 chars
6 OCR 1983 chars
7 OCR 2287 chars
8 OCR 2448 chars
9 OCR 2472 chars
10 OCR 2160 chars
11 OCR 2202 chars
12 OCR 2560 chars
13 OCR 679 chars
14 OCR 2423 chars
15 OCR 1716 chars
16 OCR 1413 chars
17 OCR 2088 chars
18 OCR 2385 chars
19 OCR 1924 chars
20 OCR 2245 chars
21 OCR 2559 chars
22 OCR 1904 chars
23 OCR 2410 chars
24 OCR 2561 chars
25 OCR 2239 chars


In [102]:
print(raw_pages[1]['text'][:1500])

۳ شیوه‌نامه اجرائی آئین‌نامه یکپارچه مقررات آموزشی دوره های تحصیلی کارذانی. کارشناسی.
کارشناسی ارشد. دکتری تخصصی بر اساس آیین‌نامه شماره 9۱۵۹۶۲ مورخ ۱۰۳۲/۱/۲۷ لازم الاجرا از
ابتدای سال تحصیلی ۱۰۲-۱4۰۳

مقدمه

به استناد ماده ۵۱ نامه شماره ۱۵۴۹۴۶ /و مورخ ۱۴۰۳/۶/۲۷ معاون محترم آموزشی وزارت علوم؛ تحقیقات و فناوری در خصوص
ابلاغ آیین نامه یکپارچه آموزشی دوره‌های تحصیلی (کاردانی/ کارشناسی؛ کارشناسی ارشد و دکتری تخصصی) و الزام تدوین
و ابلاغ شیوه نامه در خصوص موضوعاتی که در آیین‌نامه فوق اشاره نشده است و به منظور وحدت رویه در واحدهای مختلف

آموزشی دانشگاه شهید باهنر کرمان که مجری دوره های تحصیلی موضوع آیین نامه می باشند. اين شیوه‌نامه تنظیم گردیده است.
بخش نخست: مواد عمومی

ماده ۱. با استناد به ماده ۲۲ آئین نامه یکپاررچه مقررات آموزشی, انتخاب راهنمای آموزشی بر عهده رئیس بخش می باشد.
تبصره ۱: برای دانشجویان تحصیلات تکمیلی رئیس بخش موظف است برای هدایت تحصیلی دانشجو بعد از زمان پذیرش و
ثبت نام» یکی از اعضای هیات علمی مرتبط با رشته تحصیلی دانشجو را به عنوان "راهنمای آموزشی " تعیین و بر اساس سازو کا

### Native text

In [95]:
pdf_path = pdfs[4]
raw_pages = extract.extract_pdf(pdf_path)
for p in raw_pages:
    print(p["page"], "OCR" if p["was_ocr"] else "native", len(p["text"]), "chars")

راهنمای شرکت در امتحانات پایان ترم.pdf: 100%|██████████| 1/1 [00:02<00:00,  2.93s/page, native]


1 native 2270 chars


In [96]:
print(raw_pages[0]['text'])


اطالعیه 
راهنمای شرکت دانشجویان در امتحانات پایان نیمسال 
 
طبق مصوبات شورای آموزشی دانشگاه، موارد ذیل را جهت شرکت در امتحان
ا:ت پایان نیمسال مدنظر قرار دهید 
1
- کسب ا،)طالع دقیق از زمان (روز و ساعت 
 محل برگزاری و شماره صندلی هر امتحان
از طریق سیستم گلستان 
 و حضور دانشجو
 حداقل نیم ساعت قبل
از برگزاری آزمون، ضروری است. 
توجه: با توجه به عدم نصب شماره صندلی در محل تاالر و شرو
ع 
 امتحانات راس ساعت مقرر، عمال امکان دریافت و یا دسترسی به شماره
 صندلی در محل تاالر ممکن نبوده و دانشجویان محترم باید قبل از حضور در.تاالر از شماره صندلی خود مطلع شوند 
 بدیهی است عدم
اطالع دانشجو از شماره صندلی خود 
 موجب اختالل در روند برگزاری آزمون و از دست دادن زمان، اتالف وقت و یا عدم امکان
شرکت 
.درآزمون برای دانشجویان خواهد شد 
 با توجه به فشردگی بازه های برگزاری آزمون و محدودیت زمانی هر آزمون، از ورود دانشجویان
 بعد از شروع آزمون
به تاالر ممانعت بعمل آم ده و دانشجویانی که شماره صندلی خود را به
پیدا نکرده 
 باشند به خارج از تاالر راهنمائی
خواه
ن.د شد 
2
- 
 ،به همراه داشتن اصل کارت دانشجوئی برای شرکت د

### Table Detection

In [97]:
import fitz
from pipeline import tables

check_path = config.PDF_DIR / "خلاصه قوانین آموزشی.pdf"  # change to whichever file/page you're checking
check_page_num = 1  # 1-indexed

doc = fitz.open(check_path)
page = doc[check_page_num - 1]

found = tables.extract_tables(page)
print(f"{len(found)} table(s) found on page {check_page_num}")
for md in found:
    print(md)
    print("---")
doc.close()

1 table(s) found on page 1
|ﯽﺼﺼﺨﺗ يﺮﺘﮐد|ﺪﺷرا ﯽﺳﺎﻨﺷرﺎﮐ|ﯽﺳﺎﻨﺷرﺎﮐ|ه رود/ناﻮﻨﻋ|
|---|---|---|---|
|6-10|8-14|12-20|ﺣﺪاﻗﻞ و ﺣﺪاﮐﺜﺮ واﺣﺪ|
|14|12|10|ﺣﺪاﻗﻞ ﻧﻤﺮه ﻗﺎﺑﻞ ﻗﺒﻮل|
|16|14|12|ﺣﺪاﻗﻞ ﻣﻌﺪل ﻏﯿﺮ ﻣﺸﺮوﻃﯽ|
|---|2|3|ﻣﺸﺮوﻃﯽ ﻣﻨﺠﺮ ﺑﻪ اﺧﺮاﺟﯽ<br>()ﭘﯿﻮﺳﺘﻪ و ﻧﺎﭘﯿﻮﺳﺘﻪ|
|16|14|12|ﺣﺪاﻗﻞ ﻣﻌﺪل ﻓﺎرغ<br>اﻟﺘﺤﺼﯿﻠﯽ/دﻓﺎع|
|8+1<br> ﺗﺮم|4+1<br> ﺗﺮم|8+0<br> ﺗﺮم|ﺣﺪاﮐﺜﺮ ﺳﻨﻮات ﻣﺠﺎز و ارﻓﺎﻗﯽ()ﺗﺮم|
|2 ﺗﺮم|1 ﺗﺮم|2 ﺗﺮم|ﺳﻨﻮات<br>ﮐﻤﯿﺴﯿﻮن ﻣﻮارد ﺧﺎص<br>()ﺗﺮم|
|6 ﺳﺎل|3 ﺳﺎل|5 ﺳﺎل|(ﺣﺪاﮐﺜﺮ ﺳﻨﻮات ﻣﻌﺎﻓﯿﺖ ﺗﺤﺼﯿﻠﯽ )ﺳﺎل|
|---|---|14|ﺣﺪاﮐﺜﺮ اﺧﺬ واﺣﺪ ﻣﺸﺮوﻃﯿﻬﺎ|
|---|---|24|ﺣﺪاﮐﺜﺮ واﺣﺪ ﻣﻌﺪل اﻟﻒ|
|36|28-32|132-140|واﺣﺪﻫﺎي دوره|
|ﺳﻪ ﺷﺎﻧﺰدﻫﻢ ﺗﺮم|ﺳﻪ ﺷﺎﻧﺰدﻫﻢ ﺗﺮم|ﺳﻪ ﺷﺎﻧﺰدﻫﻢ ﺗﺮم|ﺗﻌﺪاد ﻏﯿﺒﺖ ﻣﻨﺠﺮ ﺑﻪ ﻧﻤﺮه ﺻﻔﺮ|
|----|اﺑﺘﺪاي ﻫﻔﺘﻪ دوم ﺗﺮم|اﺑﺘﺪاي ﻫﻔﺘﻪ دوم ﺗﺮم|(ﺗﺮﻣﯿﻢ )ﺣﺬف و اﺿﺎﻓﻪ|
|ﻫﻔﺘﻪ ﭼﻬﺎرم|ﻫﻔﺘﻪ ﭼﻬﺎرم|ﻫﻔﺘﻪ ﭼﻬﺎرم|ﺑﺮرﺳﯽ واﺣﺪﻫﺎي اﻧﺘﺨﺎﺑﯽ|
|ﻧﺪارد|ﻫﻔﺘﻪ ﺳﯿﺰدﻫﻢ ﺗﺮم|ﻫﻔﺘﻪ ﺳﯿﺰدﻫﻢ ﺗﺮم|(زﻣﺎن ﺣﺬف اﺿﻄﺮاري )ﺗﮏ درس|
|ﻫﻔﺘﻪ<br>ﭘﺎﻧﺰدﻫﻢ<br> ﺗﺮم|ﻫﻔﺘﻪ<br>ﭘﺎﻧﺰدﻫﻢ<br> ﺗﺮم|ﻫﻔﺘﻪ<br>ﭘﺎﻧﺰدﻫﻢ<br> ﺗﺮم|زﻣﺎن اﻧﺘﺨﺎب واﺣﺪ ﻣﻘﺪﻣﺎﺗﯽ ﺗﺮم ﺑﻌﺪ|
|ﻫﻔﺘﻪ ﺷﺎﻧﺰدﻫﻢ|ﻫﻔﺘﻪ ﺷﺎﻧﺰدﻫﻢ|ﻫﻔﺘﻪ ﺷﺎﻧﺰدﻫﻢ|ﺑﺮرﺳﯽ ﺑﺮﻧ

#

### OCR Module Tuning (May do later)

## Normalization

Sample normalization effect on text.

In [59]:
sample = raw_pages[1]["text"]
cleaned = normalize.clean_text(sample)
print(cleaned)

3 شیوه‌نامه اجرائی آئین‌نامه یکپارچه مقررات آموزشی دوره‌های تحصیلی کارذانی. کارشناسی. کارشناسی ارشد. دکتری تخصصی بر اساس آیین‌نامه شماره 915962 مورخ 1032 / 1 / 27 لازم‌الاجرا از ابتدای سال تحصیلی 102 - 1403

مقدمه

به استناد ماده 51 نامه شماره 154946 / و مورخ 1403 / 6 / 27 معاون محترم آموزشی وزارت علوم؛ تحقیقات و فناوری در خصوص ابلاغ آیین‌نامه یکپارچه آموزشی دوره‌های تحصیلی (کاردانی / کارشناسی؛ کارشناسی ارشد و دکتری تخصصی) و الزام تدوین و ابلاغ شیوه‌نامه در خصوص موضوعاتی که در آیین‌نامه فوق اشاره نشده است و به منظور وحدت رویه در واحدهای مختلف

آموزشی دانشگاه شهید باهنر کرمان که مجری دوره‌های تحصیلی موضوع آیین‌نامه می‌باشند. این شیوه‌نامه تنظیم گردیده است. بخش نخست: مواد عمومی

ماده 1. با استناد به ماده 22 آئین نامه یکپاررچه مقررات آموزشی, انتخاب راهنمای آموزشی بر عهده رئیس بخش می‌باشد. تبصره 1: برای دانشجویان تحصیلات تکمیلی رئیس بخش موظف است برای هدایت تحصیلی دانشجو بعد از زمان پذیرش و ثبت‌نام» یکی از اعضای هیات علمی مرتبط با رشته تحصیلی دانشجو را به عنوان «راهنمای آموزشی» تعیین و بر ا

## Chunking

Cleaning and chunking, then check chunk sizes, page numbers, and whether article numbers got picked up correctly.

In [103]:
pages = [{"page": p["page"], "text": normalize.clean_text(p["text"])} for p in raw_pages]
doc_meta = config.DOCUMENT_MANIFEST.get(pdf_path.name, {})
chunks = chunk.chunk_document(pages, doc_meta)

print("page -- Article Number -- text length")
for c in chunks:
    print(c["page"], c["article_numbers"], len(c["text"]), "chars")

page -- Article Number -- text length
2 ['1', '2', '3'] 2133 chars
2 ['3'] 2062 chars
3 ['3', '5'] 2015 chars
4 ['5'] 2040 chars
5 ['5', '10', '11'] 2036 chars
6 ['11', '16'] 2068 chars
7 ['16'] 2009 chars
8 ['16', '11'] 2145 chars
9 ['11', '17'] 2223 chars
10 ['17'] 2089 chars
10 ['17'] 2101 chars
11 ['17'] 1953 chars
12 ['17'] 1960 chars
13 ['17', '20'] 2050 chars
14 ['20', '22'] 1971 chars
15 ['22', '33'] 2000 chars
16 ['33', '36'] 2131 chars
17 ['36', '25'] 2068 chars
18 ['25'] 1967 chars
19 ['25'] 2045 chars
20 ['25', '27'] 2082 chars
20 ['27'] 2079 chars
21 ['27'] 1874 chars
22 ['27'] 1966 chars
23 ['27'] 1942 chars
24 ['27'] 1944 chars
24 ['27', '320', '33'] 2028 chars
25 ['33', '36', '35'] 1273 chars


## Full Pipeline

In [104]:
from pipeline import build_index
build_index.main()

[ainname karshenasi phd msc 1397.pdf]


ainname karshenasi phd msc 1397.pdf: 100%|██████████| 11/11 [00:08<00:00,  1.26page/s, OCR, slower]


    OCR used on page(s) [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
    9 chunks stored
[shivename 1402.pdf]


shivename 1402.pdf: 100%|██████████| 25/25 [00:27<00:00,  1.12s/page, OCR, slower]


    OCR used on page(s) [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
    28 chunks stored
[خلاصه قوانین آموزشی.pdf]


خلاصه قوانین آموزشی.pdf: 100%|██████████| 1/1 [00:00<00:00,  1.94page/s, native]


... (5 outputs left out)

مجموعه مقررات آموزشی 1402.pdf: 100%|██████████| 127/127 [04:11<00:00,  1.98s/page, native]   


    OCR used on page(s) [1, 2]
    134 table(s) found on page(s) [4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 119, 120, 121, 122, 123, 124, 125, 126, 127]
    274 chunks stored
[مقررات-آموزشی-ابلاغی-1402.pdf]


مقررات-آموزشی-ابلاغی-1402.pdf: 100%|██████████| 16/16 [00:53<00:00,  3.34s/page, OCR, slower]


    OCR used on page(s) [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
    21 chunks stored

Done. 338 chunks across 7 document(s) in 'ukask_regulations'.


## Retrieval sanity check

In [105]:
client = store.get_client()
collection = client.get_collection(config.COLLECTION_NAME)

query = "شرایط حذف و اضافه چیست؟"
query_vec = embed.embed_texts([query])
results = collection.query(query_embeddings=query_vec, n_results=10)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(meta, round(dist, 3))
    print(doc[:200])
    print("---")

{'title': 'مجموعه قوانین و آیین\u200cنامه\u200cهای آموزشی 1402', 'source_file': 'مجموعه مقررات آموزشی 1402.pdf', 'publication_date': '1402', 'article_numbers': 'unknown', 'page': 62, 'chunk_type': 'table'} 0.429
|یشزومآ یاهه مان|نیئآ و تاررقم ه|عومجم|Col 4 |Col 5 |Col 6 |Col 7 |Col 8 |<br> 58 |
|---|---|---|---|---|---|---|---|---|
||<br>|<br>|||||||
|حدهای اخذ شده|ی<br>که تعد اد وا|ند به گون<br>ه <br> ا|جو 
---
{'source_file': 'مجموعه مقررات آموزشی 1402.pdf', 'article_numbers': 'unknown', 'page': 19, 'chunk_type': 'table', 'publication_date': '1402', 'title': 'مجموعه قوانین و آیین\u200cنامه\u200cهای آموزشی 1402'} 0.431
| 15 |هتسویپان و هتس|ویپ یسانشراک|، ینادراک|یاه ه رود|یشز|ومآ یاه ه ما|نن یئآ و تاررقم هع|ومجم: ل|وا شخب|
|---|---|---|---|---|---|---|---|---|---|
|||||||||||
|ن، از غیرحضوری|داز به رایگا|از شهریه پ
---
{'source_file': 'مجموعه مقررات آموزشی 1402.pdf', 'article_numbers': 'unknown', 'publication_date': '1402', 'chunk_type': 'table', 'page': 60, 'title': 'مجموعه قوانین 